In [1]:
# Connect to the same database we just instantiated earlier:

from sqlitesearch import TextSearchIndex

sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

In [3]:
# Check how many documents are in the index:

sqlite_index.count() # 79 - as we filtered only docs for LLM zoomcamp while creating this index file

79

In [4]:
# Try a search:

results = sqlite_index.search("Can I still join the course after it started?", num_results=5)
[doc["question"] for doc in results]

['I just discovered the course. Can I still join?',
 'I missed the first homework - can I still get a certificate?',
 'Do we submit 2 projects, what does attempt 1 and 2 mean?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'I am using Azure OpenAI and I am still getting an error of Error code: 400 - {\'error\': {\'message\': "Missing required parameter: \'tools[0].function\'.", \'type\': \'invalid_request_error\', \'param\': \'tools[0].function\', \'code\': \'missing_required_parameter\'}}?']

In [5]:
# RAG with sqlitesearch
# We use the RAGBase class from rag_helper.py with this sqlitesearch index.

# Because our RAG is modular, we just swap the search index - the rest of the code stays the same:

from rag_helper import RAGBase
from openai import OpenAI

openai_client = OpenAI()

assistant = RAGBase(
    index=sqlite_index,
    llm_client=openai_client,
)

In [8]:
# This code skips both the fit call and the data loading. The index is already populated by the ingestion notebook, 
# so we just connect to the database file.

# Try it:

answer = assistant.rag("Can I still join the course after it started?")
print(answer)

# The answer should be similar to what we got with minsearch. But now the data comes from a persistent index - no fetching, no processing, 
# no indexing at startup. And we didn't have to rewrite any of the RAG logic - just swapped the index.

# The modular design splits the work cleanly:

# ingest.py handles data loading and indexing
# rag_helper.py handles the RAG pipeline
# the notebooks wire them together
# This works because sqlitesearch follows the same API as minsearch - both have a search method that takes a query, boost_dict, 
# filter_dict, and num_results. If the API were different, we'd need to subclass RAGBase and override the search method to adapt to the new backend.


Yes, you can still join the course after it started. If you want a certificate, make sure to submit your project while submissions are still being accepted.


In [9]:
# Choosing an approach
# Pick the right tool for your data:

# minsearch: single process, in-memory only, re-indexes on every startup. Use when data is small and indexing is fast.
# sqlitesearch: separate ingestion and query, file-based (SQLite), opens existing index. Use when data is large or ingestion is slow.
# Use minsearch when you can load and index the data on startup without noticeable delay. Switch to a persistent backend when ingestion takes
# too long or when you need the index to survive restarts.

# For larger production systems, use the same pattern with a different backend:

# Elasticsearch
# OpenSearch
# Qdrant (vector database)
# Weaviate (vector database)
# The architecture stays the same: one process ingests, another queries.

In [10]:
# Cleaning up
# When you're done, close the database connection:

sqlite_index.close()

# Or just let Python clean it up when the notebook kernel shuts down.